in this version we have to download data after each session and rename it like P01_smell.txt OR P01_no_smell.txt in a folder: experiment_data/ and then zip it
   

    

#Import files

In [1]:
from google.colab import files

uploaded = files.upload()

Saving SART2.zip to SART2.zip


# Extract Zip file

In [2]:
import zipfile
import os
import glob
import pandas as pd
import numpy as np

zip_filename = list(uploaded.keys())[0]
extract_folder = "experiment_data"

with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall(extract_folder)

print("Files extracted successfully.")

Files extracted successfully.


#Calculate SART2 primary outcomes

In [3]:
# Find all txt files inside the extracted folder
all_txt_files = glob.glob(
    "experiment_data/**/*.txt",
    recursive=True
)

results = []

for filepath in all_txt_files:

    filename = os.path.basename(filepath)
    name = filename.replace(".txt", "")

    # Expected file names:
    # P01_smell.txt
    # P01_no_smell.txt
    participant, condition = name.split("_", 1)

    # Read PsyToolkit data
    data = pd.read_csv(
        filepath,
        sep=r"\s+",
        header=None
    )

    data.columns = [
        "block",
        "blocknum",
        "go_nogo",
        "digit",
        "stimulus",
        "correct",
        "rt"
    ]

    # Keep only real test trials
    data = data[data["block"] == "realtest"]

    if data.empty:
        print("No realtest trials found:", filename)
        continue

    # Define trial types
    go_trials = data[data["go_nogo"] == 1]
    nogo_trials = data[data["go_nogo"] == 0]

    # Commission errors:
    # Incorrect responses on no-go trials
    commission_errors = len(
        nogo_trials[nogo_trials["correct"] == 0]
    )

    commission_error_percent = (
        commission_errors / len(nogo_trials)
    ) * 100

    # Omission errors:
    # Failure to respond on go trials
    omission_errors = len(
        go_trials[go_trials["correct"] == 0]
    )

    omission_error_percent = (
        omission_errors / len(go_trials)
    ) * 100

    # RT variability:
    # Standard deviation of RT from correct go trials only
    # Implausible RTs below 100 ms or above 1500 ms are excluded
    correct_go = go_trials[
        (go_trials["correct"] == 1) &
        (go_trials["rt"] >= 100) &
        (go_trials["rt"] <= 1500)
    ]

    rt_variability = correct_go["rt"].std()

    results.append({
        "ParticipantID": participant,
        "Condition": condition,
        "CommissionErrors": commission_error_percent,
        "OmissionErrors": omission_error_percent,
        "RTVariability": rt_variability
    })

# Create long-format dataframe
results_df = pd.DataFrame(results)

print("Long-format results:")
display(results_df)

# Convert to wide format: one row per participant
wide_df = results_df.pivot(
    index="ParticipantID",
    columns="Condition"
)

# Flatten column names
wide_df.columns = [
    f"{col[0]}_{col[1]}"
    for col in wide_df.columns
]

wide_df = wide_df.reset_index()

# Reorder and rename columns
wide_df = wide_df[[
    "ParticipantID",

    "CommissionErrors_no_smell",
    "CommissionErrors_smell",

    "OmissionErrors_no_smell",
    "OmissionErrors_smell",

    "RTVariability_no_smell",
    "RTVariability_smell"
]]

wide_df.columns = [
    "ParticipantID",

    "CommissionErrors_NoSmell",
    "CommissionErrors_Smell",

    "OmissionErrors_NoSmell",
    "OmissionErrors_Smell",

    "RTVariability_NoSmell",
    "RTVariability_Smell"
]

# Sort participants
wide_df = wide_df.sort_values("ParticipantID")

# Round final numeric output to 2 decimals
numeric_cols = wide_df.select_dtypes(include="number").columns
wide_df[numeric_cols] = wide_df[numeric_cols].round(2)

print("Final participant-level SART2 primary outcomes:")
display(wide_df)

# Save and download primary outcome dataset
primary_outcomes_file = "SART2_PrimaryOutcomes.xlsx"

wide_df.to_excel(
    primary_outcomes_file,
    index=False
)

Long-format results:


,ParticipantID,Condition,CommissionErrors,OmissionErrors,RTVariability
0,P13,smell,33.333333,2.083333,59.711618
1,P19,smell,0.000000,10.416667,97.873130
2,P15,no_smell,0.000000,0.000000,100.724600
3,P12,no_smell,0.000000,6.250000,205.424702
4,p18,smell,83.333333,2.083333,89.725494
5,P09,no_smell,16.666667,8.333333,129.290003
6,P08,no_smell,66.666667,10.416667,111.434848
7,P23,smell,16.666667,0.000000,141.085533
8,P23,no_smell,33.333333,12.500000,183.095935
9,P11,no_smell,50.000000,2.083333,168.215631


Final participant-level SART2 primary outcomes:


,ParticipantID,CommissionErrors_NoSmell,CommissionErrors_Smell,OmissionErrors_NoSmell,OmissionErrors_Smell,RTVariability_NoSmell,RTVariability_Smell
0,P04,33.33,66.67,0.00,4.17,108.79,77.33
1,P05,0.00,0.00,2.08,0.00,119.33,113.00
2,P06,33.33,16.67,4.17,0.00,55.68,60.06
3,P07,100.00,66.67,6.25,6.25,223.78,117.28
4,P08,66.67,50.00,10.42,2.08,111.43,63.06
5,P09,16.67,50.00,8.33,16.67,129.29,91.08
6,P10,33.33,0.00,4.17,0.00,138.19,66.36
7,P11,50.00,16.67,2.08,2.08,168.22,102.68
8,P12,0.00,0.00,6.25,6.25,205.42,148.47
9,P13,33.33,33.33,4.17,2.08,81.36,59.71


#Descriptive statistics table

In [4]:
descriptive_results = []

measures = {
    "Commission Error Percentage": {
        "No-Smell": "CommissionErrors_NoSmell",
        "Smell": "CommissionErrors_Smell"
    },
    "Omission Error Percentage": {
        "No-Smell": "OmissionErrors_NoSmell",
        "Smell": "OmissionErrors_Smell"
    },
    "RT Variability": {
        "No-Smell": "RTVariability_NoSmell",
        "Smell": "RTVariability_Smell"
    }
}

for measure_name, conditions in measures.items():
    for condition_name, column_name in conditions.items():

        descriptive_results.append({
            "Measure": measure_name,
            "Condition": condition_name,
            "Mean": wide_df[column_name].mean(),
            "SD": wide_df[column_name].std()
        })

descriptive_df = pd.DataFrame(descriptive_results)

# Round Mean and SD to 2 decimals
descriptive_df["Mean"] = descriptive_df["Mean"].round(2)
descriptive_df["SD"] = descriptive_df["SD"].round(2)

display(descriptive_df)

descriptive_file = "SART2_DescriptiveStatistics.xlsx" #Table 5.1

descriptive_df.to_excel(
    descriptive_file,
    index=False
)

,Measure,Condition,Mean,SD
0,Commission Error Percentage,No-Smell,32.35,27.93
1,Commission Error Percentage,Smell,31.37,28.80
2,Omission Error Percentage,No-Smell,4.53,3.84
3,Omission Error Percentage,Smell,3.80,4.44
4,RT Variability,No-Smell,131.12,43.64
5,RT Variability,Smell,93.75,29.47


#Shapiro–Wilk normality test for paired difference scores

In [5]:
from scipy.stats import shapiro

normality_results = []

comparisons = {
    "Commission Error Percentage": {
        "Smell": "CommissionErrors_Smell",
        "No-Smell": "CommissionErrors_NoSmell"
    },
    "Omission Error Percentage": {
        "Smell": "OmissionErrors_Smell",
        "No-Smell": "OmissionErrors_NoSmell"
    },
    "RT Variability": {
        "Smell": "RTVariability_Smell",
        "No-Smell": "RTVariability_NoSmell"
    }
}

for measure_name, cols in comparisons.items():

    difference_scores = (
        wide_df[cols["Smell"]] - wide_df[cols["No-Smell"]]
    ).dropna()

    if len(difference_scores) >= 3:
        stat, p_value = shapiro(difference_scores)

        normality_results.append({
            "Measure": measure_name,
            "Shapiro_W": stat,
            "p_value": p_value,
            "Normality_Assumption": "Met" if p_value >= 0.05 else "Not met"
        })

    else:
        normality_results.append({
            "Measure": measure_name,
            "Shapiro_W": np.nan,
            "p_value": np.nan,
            "Normality_Assumption": "Not enough data"
        })

normality_df = pd.DataFrame(normality_results)

# Round Shapiro-W to 2 decimals and p-value to 3 decimals
normality_df["Shapiro_W"] = normality_df["Shapiro_W"].round(2)
normality_df["p_value"] = normality_df["p_value"].round(3)

display(normality_df)

normality_file = "SART2_NormalityResults.xlsx" #Table F.1

normality_df.to_excel(
    normality_file,
    index=False
)

,Measure,Shapiro_W,p_value,Normality_Assumption
0,Commission Error Percentage,0.90,0.056,Met
1,Omission Error Percentage,0.98,0.929,Met
2,RT Variability,0.96,0.568,Met


#Paired t-test

In [6]:
from scipy.stats import ttest_rel

ttest_results = []

for measure_name, cols in comparisons.items():

    paired_data = wide_df[
        [cols["Smell"], cols["No-Smell"]]
    ].dropna()

    smell_values = paired_data[cols["Smell"]]
    no_smell_values = paired_data[cols["No-Smell"]]

    difference_scores = smell_values - no_smell_values

    if len(paired_data) >= 2:

        t_stat, p_value = ttest_rel(
            smell_values,
            no_smell_values
        )

        diff_sd = difference_scores.std(ddof=1)

        if diff_sd == 0:
            cohens_d = np.nan
        else:
            cohens_d = difference_scores.mean() / diff_sd

        ttest_results.append({
            "Measure": measure_name,
            "t_value": round(t_stat, 2),
            "p_value": round(p_value, 3),
            "Cohens_d": round(cohens_d, 2)
        })

    else:

        ttest_results.append({
            "Measure": measure_name,
            "t_value": np.nan,
            "p_value": np.nan,
            "Cohens_d": np.nan
        })

ttest_results_df = pd.DataFrame(ttest_results)

display(ttest_results_df)

ttest_file = "SART2_PairedTTestResults.xlsx" #Table 5.2

ttest_results_df.to_excel(
    ttest_file,
    index=False
)

,Measure,t_value,p_value,Cohens_d
0,Commission Error Percentage,-0.12,0.906,-0.03
1,Omission Error Percentage,-0.57,0.576,-0.14
2,RT Variability,-5.39,0.000,-1.31


#Downloading Results

In [7]:
from google.colab import files

files.download("SART2_PrimaryOutcomes.xlsx")              #needed for master dataset
files.download("SART2_DescriptiveStatistics.xlsx")        #Table 5.2
files.download("SART2_NormalityResults.xlsx")             #Table F.1
files.download("SART2_PairedTTestResults.xlsx")           #Table 5.3

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>